# Serve the real Qwen3.5-9B for RevOps Agent (vLLM)
Runtime: GPU with >= 24 GB (Colab A100/L4, Kaggle 2xT4 uses fp16 + tensor parallel). Downloads ~18 GB of real weights from Hugging Face.
Then run the last cell to get a public URL and put it in `apps/ai/.env` as `LLM_BASE_URL=<url>/v1`.

In [ ]:
!nvidia-smi
!pip -q install -U vllm huggingface_hub hf_transfer

In [ ]:
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
os.environ['LLM_API_KEY']='CHANGE-ME-dev-llm-key'
!huggingface-cli download Qwen/Qwen3.5-9B   # resumable real download (~18 GB)

In [ ]:
import subprocess, os
ngpu = len(subprocess.check_output(['nvidia-smi','-L']).decode().strip().splitlines())
dtype = 'auto'
cmd = ['vllm','serve','Qwen/Qwen3.5-9B','--host','0.0.0.0','--port','8001','--api-key',os.environ['LLM_API_KEY'],
       '--dtype',dtype,'--tensor-parallel-size',str(ngpu),'--max-model-len','16384','--gpu-memory-utilization','0.90',
       '--enable-prefix-caching','--served-model-name','Qwen/Qwen3.5-9B']
server = subprocess.Popen(cmd, stdout=open('vllm.log','w'), stderr=subprocess.STDOUT)
print('started; tail vllm.log until it says Application startup complete')

In [ ]:
!tail -n 20 vllm.log

In [ ]:
# public URL (Cloudflare quick tunnel, no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8001'], stdout=open('tunnel.log','w'), stderr=subprocess.STDOUT)
import time; time.sleep(8)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1